# Lecture 15 — Masks: Asking Questions of a Year of Data
### EES 3350/5350 · Python in Earth Science · Friday, October 09, 2026

On Wednesday we found that 2025 at the Nashville airport ran about **2 °F warmer** and about **4 inches wetter** than the 1991–2020 normal. A yearly average hides a lot, though. Was it warm because the winter was mild, or because the summer was brutal? On how many days did it actually rain?

Questions like these all have the same shape: *on which days was something true?* For three weeks you've answered that shape with a loop and an `if`. Today you'll learn the NumPy way: a **mask**.

**Learning objectives for today**
- make a **boolean array** with a comparison, and count its `True`s with `np.sum()`
- use a mask to **select** only the elements you want
- combine conditions with `&` and `|` (and know why `and` and `or` fail)
- find **which day** something happened with `.argmax()` and `.argmin()`
- answer yes/no questions with `.any()` and `.all()`
- clean a fill value without damaging your raw data, and explain why changing a slice of an array can change the original
- recognize a question that still needs a loop

---
## Part 0 : get today's notebook

Open a **terminal** in ExploreHub and pull, just like Wednesday:
```
cd ~/course-materials
git pull
ls week07
```
You should see `15_numpy_masks.ipynb` next to Wednesday's files. Then copy it into your own folder and rename it:
```
cp week07/15_numpy_masks.ipynb ~/LastName_EES3350/lectures/15_numpy_masks_LastName.ipynb
```

The data file stays where it is, in `course-materials/week07`. The **relative path** below walks there from your `lectures` folder: up out of `lectures`, up out of `LastName_EES3350`, then down into `course-materials/week07`.

In [1]:
import numpy as np

data_path = "../../course-materials/week07/nashville_daily_2025.csv"
data = np.loadtxt(data_path, delimiter=",")

month = data[:, 0]
day   = data[:, 1]
tmax  = data[:, 2]
tmin  = data[:, 3]
prcp  = data[:, 4]
snow  = data[:, 5]

print(data.shape)

(365, 6)


Same lines as Wednesday, with one addition: today we also pull out **columns 0 and 1**, `month` and `day`, because today's questions need dates.

*Data: NOAA National Centers for Environmental Information (NCEI), Global Historical Climatology Network – Daily (GHCN-Daily), station USW00013897 (Nashville Intl AP, TN), January 1 – December 31, 2025, retrieved October 6, 2026 via the [NCEI Access Data Service](https://www.ncei.noaa.gov/support/access-data-service-api-user-documentation). Normals are from NOAA NCEI U.S. Climate Normals 1991–2020 for the same station, the same values you used in Labs 04 and 05.*

---
## Part 1 : comparisons make boolean arrays

A freezing night is one with a low of **32 °F or colder**. You already know how to count them with a loop: a counter, an `if`, and `+= 1`.

In [2]:
freeze_count = 0

for t in tmin:
    if t <= 32:
        freeze_count += 1

print(freeze_count)

65


That works. Now watch what a comparison does on the whole array at once. We'll look at the first week of January so the output fits on the screen:

In [3]:
print(tmin[:7])
print(tmin[:7] <= 32)

[28. 25. 30. 29. 34. 27. 27.]
[ True  True  True  True False  True  True]


#### What just happened?
On Wednesday, math between an array and a number happened **element-wise**. Comparisons work the same way. `tmin <= 32` checks every element and hands back a new array of `True` and `False` values, one per day:

|**position**|**`tmin[i]` (°F)**|**`tmin[i] <= 32`**|
|--:|--:|:--|
|0|28|`True`|
|1|25|`True`|
|2|30|`True`|
|3|29|`True`|
|4|34|`False`|
|5|27|`True`|
|6|27|`True`|

An array of `True`/`False` values is called a **boolean array**. Its dtype is `bool`:

In [4]:
is_freezing = tmin <= 32

print(is_freezing.shape)
print(is_freezing.dtype)

(365,)
bool


It's the same shape as `tmin`, so position 0 still means January 1. Every comparison operator from Week 1 works: `<`, `<=`, `>`, `>=`, `==`, `!=`.

#### Counting the `True`s
Python treats `True` as **1** and `False` as **0** when you do math with them:

In [5]:
print(True + True + False)

2


So adding up a boolean array **counts** its `True`s. That's the whole counter loop in one line:

In [6]:
freeze_nights = np.sum(is_freezing)

print(f"Freezing nights in 2025: {freeze_nights}")
print(f"Normal (1991-2020):      66.2")

Freezing nights in 2025: 65
Normal (1991-2020):      66.2


2025 had a completely ordinary number of freezing nights. So the warm year **wasn't** a mild winter. Let's look at the other end.

<div class="alert alert-info">

### Exercise 1 : counting days

Each answer should take **one line** of NumPy. Compare each count to its normal.

1. How many days reached **90 °F or hotter**? (normal: **54.2**)
2. NOAA counts a day as a "precipitation day" if it got **at least 0.01 in**. How many precipitation days did 2025 have? (normal: **123.1**, the sum of the 12 monthly `days_with_precip` normals from Worksheet Week 02)
3. How many days got **1 inch or more**?
4. How many days hit **100 °F or hotter**?

</div>

In [ ]:
# your answer here
#1
print(f"90 F days:          {____________________}  (normal 54.2)")

#2
print(f"Precipitation days: {____________________} (normal 123.1)")

#3
print(f"1-inch days:        {____________________}")

#4
print(f"100 F days:         {____________________}")

<details>
<summary style="color:green;"><strong>Answer.</strong></summary>

```python
print(f"90 F days:          {np.sum(tmax >= 90)}  (normal 54.2)")
print(f"Precipitation days: {np.sum(prcp >= 0.01)} (normal 123.1)")
print(f"1-inch days:        {np.sum(prcp >= 1)}")
print(f"100 F days:         {np.sum(tmax >= 100)}")
```
---
```
90 F days:          79  (normal 54.2)
Precipitation days: 118 (normal 123.1)
1-inch days:        12
100 F days:         1
```

There's the warm year: **79** days at 90 °F or more, about **25 more** than normal, with an ordinary winter. And 2025 was ~4 inches wetter than normal while having *fewer* precipitation days than normal. Hold onto that; Part 2 explains it.

`np.sum(mask)` and `mask.sum()` do the same thing, just like `.sum()` on Wednesday.

</details>

#### Aside: what can a count *not* see?
Last January (2026, so not in this file), an ice storm coated Nashville in nearly an inch of ice, and about 232,000 NES customers lost power at the peak ([Axios Nashville, 2026](https://www.axios.com/local/nashville/2026/01/27/ice-storm-slams-nashville-thousands-still-powerless)). Ice storms need **precipitation** to fall when it's **freezing**. Here's a sneak peek at Part 3: `&` means "both are true on the same day."

In [7]:
print(np.sum((prcp > 0) & (tmin <= 32)))
print(np.sum((prcp > 0) & (tmax <= 32)))

16
2


<div class="alert alert-warning">

**Question:** In 2025, 16 days had precipitation with a low at or below freezing, but only 2 stayed at or below 32 °F all day. Would either count have flagged last January's ice storm? What *can't* this file tell you about a day? (Look at the columns: is there one for ice?)

</div>

---
## Part 2 : masks select data

A boolean array can do more than count. Put it **inside the square brackets** and NumPy keeps only the elements where the mask is `True`:

In [8]:
week_lows = tmin[:7]
week_freezing = week_lows <= 32

print(week_lows)
print(week_freezing)
print(week_lows[week_freezing])

[28. 25. 30. 29. 34. 27. 27.]
[ True  True  True  True False  True  True]
[28. 25. 30. 29. 27. 27.]


#### What just happened?
NumPy lines the mask up with the array, position by position, and keeps the matches:

|**position**|0|1|2|3|4|5|6|
|:--|--:|--:|--:|--:|--:|--:|--:|
|`week_lows`|28|25|30|29|34|27|27|
|`week_freezing`|`True`|`True`|`True`|`True`|`False`|`True`|`True`|
|kept?|✓|✓|✓|✓| |✓|✓|

January 5 (34 °F) dropped out, so the result has 6 elements instead of 7. A boolean array used this way is called a **mask**: like masking tape, it covers up the parts you don't want.

You've written this as a loop before: `if` + `.append()` into a new list. Here are both side by side for the whole year:

In [9]:
# the loop way
rainy_list = []
for p in prcp:
    if p >= 0.01:
        rainy_list.append(p)

# the mask way
rainy = prcp[prcp >= 0.01]

print(len(rainy_list), len(rainy))

118 118


Read `prcp[prcp >= 0.01]` out loud as **"`prcp`, where `prcp` is at least 0.01."**

#### Which average answers the question?

In [10]:
print(f"Average precipitation, all 365 days:     {prcp.mean():.2f} in")
print(f"Average precipitation, on days with rain: {rainy.mean():.2f} in")

Average precipitation, all 365 days:     0.15 in
Average precipitation, on days with rain: 0.47 in


<div class="alert alert-warning">

**Question 1:** Both numbers are correct, but they answer different questions. Which one answers *"When it rains in Nashville, how much does it rain?"* Which one would you use to estimate the year's total?

</div>

#### The mask doesn't have to come from the same array
`month` and `tmax` line up day by day, so a mask made from `month` can select from `tmax`. This picks out every high in December:

In [13]:
dec_highs = tmax[month == 12]

print(len(dec_highs))
print(f"December average high: {dec_highs.mean():.1f} F (normal 52.2 F)")

31
December average high: 53.2 F (normal 52.2 F)


Read it as **"`tmax`, where `month` is 12."** No more counting days to find index 334!

<div class="alert alert-info">

### Exercise 2 : selecting with masks

1. What was the average **low** on the freezing nights only?
2. What was the average **high** in **July**? Compare it to the July normal of **90.9 °F**.
3. Make `big_rain`, the precipitation on the days with **1 inch or more**. How many inches fell on those days in total? What **percent** of the year's precipitation is that?

</div>

In [ ]:
# your answer here
# 1
print(f"Average low on freezing nights: {___________________:.1f} F")

# 2
print(f"July average high: {___________________:.1f} F (normal 90.9 F)")

# 3
big_rain = prcp[prcp >= 1]
print(f"Rain on 1-inch days: {___________________:.2f} in")
print(f"Share of the year:   {___________________:.0f}%")

<details>
<summary style="color:green;"><strong>Answer.</strong></summary>

```python
# 1
print(f"Average low on freezing nights: {tmin[tmin <= 32].mean():.1f} F")

# 2
print(f"July average high: {tmax[month == 7].mean():.1f} F (normal 90.9 F)")

# 3
big_rain = prcp[prcp >= 1]
print(f"Rain on 1-inch days: {big_rain.sum():.2f} in")
print(f"Share of the year:   {big_rain.sum() / prcp.sum() * 100:.0f}%")
```
---
```
Average low on freezing nights: 24.5 F
July average high: 94.0 F (normal 90.9 F)
Rain on 1-inch days: 24.66 in
Share of the year:   45%
```

For #3: just **12 days** delivered almost **half** of 2025's precipitation. That's how a year can be wetter than normal with fewer rainy days than normal: the rain came in big storms. The biggest of them came on April 3, the start of a slow-moving storm system that the National Weather Service called historic: from April 2 to 6 it dropped 12 to 16 inches on parts of western Kentucky and flooded the lower Ohio Valley ([NWS Paducah, 2025](https://www.weather.gov/pah/April2025Flooding)). Nashville's share, `prcp[91:96].sum()`, was 6.32 in, more than a normal April.

</details>

---
## Part 3 : combining conditions

Hot days are bad. Hot days **with no rain** dry out soils and stress crops. Let's count days that were 90 °F or hotter **and** completely dry. You've combined conditions with `and` since Week 3, so try it:

In [14]:
hot_and_dry = (tmax >= 90) and (prcp == 0)

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

#### Reading the error
Start at the bottom: a `ValueError`. *The truth value of an array with more than one element is ambiguous.*

`and` asks a yes/no question about the thing on each side: "Is the left side true? Is the right side true?" One number can be true or false. But is an array of 365 `True`s and `False`s "true"? Python can't decide, so it refuses. (The error even suggests `a.any()` and `a.all()`. We'll meet those in Part 4.)

What we want is element-wise: pair each day's two answers and combine *those*. NumPy uses different symbols for that:

|**you want**|**one value (Weeks 3–6)**|**element-wise on arrays**|
|:--|:--|:--|
|both true|`and`|`&`|
|either true|`or`|`\|`|
|flip it|`not`|`~`|

In [15]:
hot_and_dry = (tmax >= 90) & (prcp == 0)

print(f"Hot, dry days: {np.sum(hot_and_dry)}")

Hot, dry days: 56


<div class="alert alert-danger">

**Common Error:** with arrays, use `&` and `|` instead of `and` and `or`, and **put each condition in its own parentheses**: `(tmax >= 90) & (prcp == 0)`, never `tmax >= 90 & prcp == 0`. If you see *"truth value of an array... is ambiguous"*, look for an `and`, `or`, or `if` that got an array. If you see an error mentioning *"bitwise_and"*, you're missing parentheses.

</div>

#### A range of months
"Summer" is June through August: the month is at least 6 **and** at most 8.

In [16]:
is_summer = (month >= 6) & (month <= 8)

print(np.sum(is_summer))
print(f"Summer average high: {tmax[is_summer].mean():.1f} F")

92
Summer average high: 91.3 F


That's the same 91.3 °F you got on Wednesday with `tmax[151:243]`, but this time nobody had to look up that June 1 is index 151.

<div class="alert alert-info">

### Exercise 3 : combined masks

1. How many of the year's 90 °F days were **in summer**? How many were **outside** summer? (Hint: you can combine `is_summer` with another condition, and `~is_summer` flips it.)
2. Heat stress builds up two ways: a hot afternoon, or a night that never cools off. Count the **heat-stress days**: a high of **90 °F or hotter** *or* a low of **75 °F or warmer**. Before you run it, predict: is the answer 79 + 23 = 102?
3. Find the heat-stress day(s) that had a warm night but **not** a 90 °F afternoon. Print the date and the high. (Use `&` and `~`.)

</div>

In [ ]:
# your answer here
# 1
print(f"90 F days in summer:      {___________________}")
print(f"90 F days outside summer: {___________________}")

# 2
hot_day = ___________________
warm_night = ___________________
heat_stress = ___________________
print(f"Hot days: {___________________}, warm nights: {___________________}")
print(f"Heat-stress days: {___________________}")

# 3
night_only = ___________________
print(month[___________________], day[___________________], tmax[___________________])

<details>
<summary style="color:green;"><strong>Answer.</strong></summary>

```python
# 1
print(f"90 F days in summer:      {np.sum(is_summer & (tmax >= 90))}")
print(f"90 F days outside summer: {np.sum(~is_summer & (tmax >= 90))}")

# 2
hot_day = tmax >= 90
warm_night = tmin >= 75
heat_stress = hot_day | warm_night
print(f"Hot days: {np.sum(hot_day)}, warm nights: {np.sum(warm_night)}")
print(f"Heat-stress days: {np.sum(heat_stress)}")

# 3
night_only = warm_night & ~hot_day
print(month[night_only], day[night_only], tmax[night_only])
```
---
```
90 F days in summer:      65
90 F days outside summer: 14
Hot days: 79, warm nights: 23
Heat-stress days: 80
[8.] [1.] [83.]
```

For #1: 14 of the 90 °F days came in **September**, which is outside meteorological summer. The heat didn't stop on August 31.

For #2: not 102, only **80**. 22 of the 23 warm nights followed a 90 °F afternoon, and `|` counts each of those days **once**. Adding two separate counts would have counted them twice. That's the difference between `|` and `+`.

For #3: August 1 topped out at only 83 °F, but the low never got below 75 °F.

</details>

---
## Part 4 : which day? and yes/no questions

#### `.argmax()` and `.argmin()`
On Wednesday, `tmax.max()` told us the hottest high was 101 °F. But **when** was it? In Lecture 13 you answered "when" questions with the best-so-far pattern: track the best value **and** its position as you loop. NumPy's version is `.argmax()`, which returns the **index** of the biggest value ("**arg**ument of the **max**").

In [17]:
i = tmax.argmax()

print(i)
print(tmax[i])
print(f"Hottest day: {month[i]:.0f}/{day[i]:.0f}, high {tmax[i]:.0f} F")

230
101.0
Hottest day: 8/19, high 101 F


#### What just happened?

|**what you type**|**what it gives you**|**for `tmax`**|
|:--|:--|:--|
|`tmax.max()`|the biggest **value**|101.0|
|`tmax.argmax()`|the **index** of the biggest value|230|
|`month[230]`, `day[230]`|the date at that index|8, 19 → August 19|

The index is the useful part: because every column lines up, one index unlocks the date **and** every other measurement that day. `.argmin()` does the same for the smallest value. `:.0f` in the f-string drops the `.0` off the month and day, since they were read in as floats.

<div class="alert alert-info">

### Exercise 4 : find the day

For each one, print the date and the value.

1. The **coldest night** of the year.
2. The **wettest day** of the year.
3. The day with the **biggest swing** between high and low. (Make the daily range array first. You made one on Wednesday.)

</div>

In [ ]:
# your answer here
# 1
i = __________
print(f"Coldest night: {__________:.0f}/{__________:.0f}, low {__________:.0f} F")

# 2
i = prcp.argmax()
print(f"Wettest day: {__________:.0f}/{__________:.0f}, {__________:.2f} in")

# 3
daily_range = __________
i = __________
print(f"Biggest swing: {__________:.0f}/{__________:.0f}, {__________:.0f} F to {__________:.0f} F (__________:.0f} F)")

<details>
<summary style="color:green;"><strong>Answer.</strong></summary>

```python
# 1
i = tmin.argmin()
print(f"Coldest night: {month[i]:.0f}/{day[i]:.0f}, low {tmin[i]:.0f} F")

# 2
i = prcp.argmax()
print(f"Wettest day: {month[i]:.0f}/{day[i]:.0f}, {prcp[i]:.2f} in")

# 3
daily_range = tmax - tmin
i = daily_range.argmax()
print(f"Biggest swing: {month[i]:.0f}/{day[i]:.0f}, {tmax[i]:.0f} F to {tmin[i]:.0f} F ({daily_range[i]:.0f} F)")
```
---
```
Coldest night: 1/20, low 8 F
Wettest day: 4/3, 3.86 in
Biggest swing: 2/26, 80 F to 38 F (42 F)
```

For #1: January **22** also dropped to 8 °F. When there's a tie, `.argmin()` and `.argmax()` return the **first** one, exactly like a best-so-far loop that uses `<` instead of `<=`. Use `np.sum(tmin == tmin.min())` to check for ties.

</details>

#### `.any()` and `.all()`
In Lecture 13 you used a **flag variable** to answer yes/no questions: start with `found = False`, loop, and flip it to `True` if you see a match. Boolean arrays have that built in:

|**what you type**|**the question it answers**|
|:--|:--|
|`mask.any()`|Is **at least one** element `True`?|
|`mask.all()`|Is **every** element `True`?|

Did **any** night in July drop below 60 °F?

In [18]:
july_lows = tmin[month == 7]

print((july_lows < 60).any())
print(f"Coolest July night: {july_lows.min():.0f} F")

False
Coolest July night: 69 F


No: July never got below 69 °F at night. Was **every** July afternoon at least 90 °F?

In [19]:
july_highs = tmax[month == 7]

print((july_highs >= 90).all())

False


Almost, but not quite. Count the exceptions to see how close it was:

In [20]:
print(np.sum(july_highs < 90))

1


<div class="alert alert-info">

### Exercise 5 : yes or no?

Answer each with `.any()` or `.all()`, then add one more line that gives the detail.

1. Was **every** day in **August** at least **80 °F**?
2. Did **any** day in **December** reach **70 °F**? If so, how many, and which days of the month? (Remember Christmas from Wednesday?)
3. Did it **snow** on any day after March 1? (`snow` is in inches.)

</div>

In [ ]:
# your answer here
# 1
aug_highs = __________________
print(__________________)
print(f"Coolest August high: {__________________:.0f} F")

# 2
dec_warm = __________________
print(__________________)
print(__________________,__________________)

# 3
print(__________________)

<details>
<summary style="color:green;"><strong>Answer.</strong></summary>

```python
# 1
aug_highs = tmax[month == 8]
print((aug_highs >= 80).all())
print(f"Coolest August high: {aug_highs.min():.0f} F")

# 2
dec_warm = (month == 12) & (tmax >= 70)
print(dec_warm.any())
print(np.sum(dec_warm), day[dec_warm])

# 3
print((snow[month >= 3] > 0).any())
```
---
```
True
Coolest August high: 81 F
True
5 [23. 24. 25. 27. 28.]
False
```

For #2: five December days hit 70 °F or warmer, all of them between December 23 and 28. For #3, all of 2025's snow fell in January and February.

</details>

---
## Part 5 : cleaning bad data — views vs. copies

The NOAA file we're using has already been quality-checked. Raw data straight from an instrument usually hasn't. Sensors fail, batteries die, and loggers record a **fill value**, a placeholder like `-9999` that means "no reading." Many instrument and data files use one. If you don't catch it, one `-9999` wrecks every average you compute.

Here's what the first week of January might look like straight off a station logger whose sensor failed on January 4:

In [21]:
raw_highs = np.array([44., 49, 44, -9999, 56, 56, 32])

print(raw_highs.mean())

-1388.2857142857142


An average high of about −1,388 °F. Let's clean it. You're only studying **January 1–5**, so slice those days out, then use a mask to replace the fill value with `np.nan`, NumPy's way of writing "not a number" (missing):

In [22]:
jan1_5 = raw_highs[0:5]
jan1_5[jan1_5 == -9999] = np.nan

print(jan1_5)

[44. 49. 44. nan 56.]


That middle line is new: a mask on the **left** side of `=`. It means "everywhere this mask is `True`, put this value." It's the same mask as always, but now it's choosing where to *write* instead of what to *read*.

Now check the raw data, which you meant to leave alone:

In [23]:
print(raw_highs)

[44. 49. 44. nan 56. 56. 32.]


#### What just happened?
The raw data changed too! A slice of an array is a **view**: a new name for a *window into the same numbers*, not a copy of them. NumPy does this on purpose, because copying millions of numbers every time you slice would be slow. But it means "cleaning" a slice quietly edits your original. In research, that's how raw data gets lost.

Lists don't do this. A slice of a list is a real copy:

In [24]:
raw_list = [44, 49, 44, -9999, 56, 56, 32]

jan1_5_list = raw_list[0:5]
jan1_5_list[3] = 0

print(jan1_5_list)
print(raw_list)

[44, 49, 44, 0, 56]
[44, 49, 44, -9999, 56, 56, 32]


When you want an independent array, ask for one with `.copy()`, the same method you used on lists in Lecture 04:

In [25]:
raw_highs = np.array([44., 49, 44, -9999, 56, 56, 32])

jan1_5 = raw_highs[0:5].copy()
jan1_5[jan1_5 == -9999] = np.nan

print(jan1_5)
print(raw_highs)

[44. 49. 44. nan 56.]
[   44.    49.    44. -9999.    56.    56.    32.]


The raw data is safe. One more thing: `nan` is contagious. Any math that touches a `nan` gives back `nan`, so NumPy has "skip the missing values" versions of its functions:

In [26]:
print(jan1_5.mean())
print(np.nanmean(jan1_5))

nan
48.25


|**you write**|**list**|**array**|
|:--|:--|:--|
|`b = a`|same list|same array|
|`b = a[0:5]`|**copy**|**view** (changes reach `a`)|
|`b = a[0:5].copy()`|copy|copy|
|`b = a[mask]`|*(n/a)*|copy|

<div class="alert alert-success">

**Tip:** you've been using views all week without knowing it. `tmax = data[:, 2]` is a view into `data`. That's fine as long as you only *read* from it. Before you **change values** (replacing fill values, fixing a bad reading), make a `.copy()` and clean the copy. Keep your raw data raw.

</div>

---
## Part 6 : where loops still win

Masks answer questions about **individual days**. Some questions are about **runs of days in a row**, and for those, a mask can't do it in one line.

**What was the longest dry streak of 2025?** A dry day has `prcp == 0`. We need to walk through the year **in order**, keeping a running count of dry days in a row and resetting it to zero whenever it rains. That's a counter plus the best-so-far pattern from Lecture 13. Plan first, in comments:

In [27]:
# plan:
# - current = length of the dry streak we're in right now
# - longest = longest streak seen so far, and end_i = where it ended
# - for each day, in order (I need the index, so use enumerate):
#     - dry?  add 1 to current
#     - rain? reset current to 0
#     - if current beats longest, update longest and end_i

current = 0
longest = 0
end_i = 0

for i, p in enumerate(prcp):
    if p == 0:
        current += 1
    else:
        current = 0

    if current > longest:
        longest = current
        end_i = i

start_i = end_i - longest + 1

print(f"Longest dry streak: {longest} days")
print(f"From {month[start_i]:.0f}/{day[start_i]:.0f} to {month[end_i]:.0f}/{day[end_i]:.0f}")

Longest dry streak: 12 days
From 11/8 to 11/19


#### What just happened?
Here's the loop crossing November 7–20, the end of that streak:

|**date**|**`prcp`**|**dry?**|**`current`**|**`longest`**|
|:--|--:|:--|--:|--:|
|Nov 7|0.43|no|0|11|
|Nov 8|0.00|yes|1|11|
|Nov 9|0.00|yes|2|11|
|...|...|...|...|...|
|Nov 18|0.00|yes|11|11|
|Nov 19|0.00|yes|12|**12**|
|Nov 20|0.61|no|0|12|

`longest` was already 11 from an earlier dry spell, so it didn't change until `current` finally beat it on November 19. (On November 18 the two were tied, and `>` doesn't update on a tie.) Notice the masks are still useful here: `prcp == 0` is the same question we've asked all day. The **loop** is what remembers what happened *yesterday*.

The start of the streak comes from arithmetic: if a 12-day streak ended at index `end_i`, it started 11 days earlier.

<div class="alert alert-info">

### Exercise 6 : the heat wave, from scratch

What was the longest streak of days **in a row** with a high of **90 °F or hotter**? Print its length and its first and last dates.

**Write your plan as comments first**, then write the code underneath. Try it without scrolling up!

</div>

In [ ]:
# your plan and answer here

<details>
<summary style="color:green;"><strong>Answer.</strong></summary>

```python
# plan:
# - current = length of the hot streak right now; longest and end_i = best so far
# - for each day in order: hot? add 1, otherwise reset to 0
# - if current beats longest, save it and where it ended

current = 0
longest = 0
end_i = 0

for i, t in enumerate(tmax):
    if t >= 90:
        current += 1
    else:
        current = 0

    if current > longest:
        longest = current
        end_i = i

start_i = end_i - longest + 1

print(f"Longest 90 F streak: {longest} days")
print(f"From {month[start_i]:.0f}/{day[start_i]:.0f} to {month[end_i]:.0f}/{day[end_i]:.0f}")
```
---
```
Longest 90 F streak: 30 days
From 7/2 to 7/31
```

Every single day from July 2 to July 31 hit 90 °F. That's why `(july_highs >= 90).all()` in Part 4 came out `False` by just **one** day: July 1 topped out at 88 °F. The July normal is 18.2 days at 90 °F or more; 2025 had 30.

</details>

<div class="alert alert-custom" style="background-color:#e8daef; border:1px solid #d2b4de; color:#6c3483; padding:10px; border-radius:4px;">

### Bonus: the growing season

Gardeners in Nashville care about two dates: the **last spring freeze** and the **first fall freeze**. The days between them are the frost-free **growing season**.

`np.where(mask)` hands back the **indices** where a mask is `True`. It returns them wrapped in a tuple, so add `[0]` to get the array itself:
```python
freeze_i = np.where(tmin <= 32)[0]
```

1. Make `freeze_i` and print it. Notice it's in order, and that there's a big jump in the middle of it.
2. Index 181 is July 1. Use a mask on `freeze_i` to split it into `spring_i` (before July 1) and `fall_i` (after).
3. The last spring freeze is the **last** element of `spring_i`, and the first fall freeze is the **first** element of `fall_i`. Print both dates.
4. How many frost-free days were there between them?
5. Combine a loop with a mask: for each month 1 through 12, count the freezing nights in that month, and print it next to the Lab 05 normal below.

```python
freeze_normal = [18.8, 14.2, 7.4, 0.9, 0.0, 0.0, 0.0, 0.0, 0.0, 0.7, 8.4, 15.8]
```

</div>

In [ ]:
# your answer here

<details>
<summary style="color:green;"><strong>Answer.</strong></summary>

```python
# 1
freeze_i = np.where(tmin <= 32)[0]
print(freeze_i)

# 2
spring_i = freeze_i[freeze_i < 181]
fall_i = freeze_i[freeze_i > 181]

# 3
last_spring = spring_i[-1]
first_fall = fall_i[0]
print(f"Last spring freeze: {month[last_spring]:.0f}/{day[last_spring]:.0f}")
print(f"First fall freeze:  {month[first_fall]:.0f}/{day[first_fall]:.0f}")

# 4
print(f"Frost-free days: {first_fall - last_spring - 1}")

# 5
freeze_normal = [18.8, 14.2, 7.4, 0.9, 0.0, 0.0, 0.0, 0.0, 0.0, 0.7, 8.4, 15.8]
for mo, normal in zip(range(1, 13), freeze_normal):
    count = np.sum((month == mo) & (tmin <= 32))
    print(f"Month {mo:2d}: {count:2d} freezing nights (normal {normal})")
```
---
```
[  0   1   2   3   5   6   7   8   9  10  11  12  13  14  15  16  18  19
  20  21  22  23  24  27  40  43  44  46  47  48  49  50  51  52  53  54
  59  60  61  79 313 314 330 331 332 333 334 335 336 337 339 340 341 342
 344 346 347 348 349 352 353 355 362 363 364]
Last spring freeze: 3/21
First fall freeze:  11/10
Frost-free days: 233
Month  1: 24 freezing nights (normal 18.8)
Month  2: 12 freezing nights (normal 14.2)
Month  3:  4 freezing nights (normal 7.4)
Month  4:  0 freezing nights (normal 0.9)
Month  5:  0 freezing nights (normal 0.0)
Month  6:  0 freezing nights (normal 0.0)
Month  7:  0 freezing nights (normal 0.0)
Month  8:  0 freezing nights (normal 0.0)
Month  9:  0 freezing nights (normal 0.0)
Month 10:  0 freezing nights (normal 0.7)
Month 11:  6 freezing nights (normal 8.4)
Month 12: 19 freezing nights (normal 15.8)
```

For #4: subtracting two indices counts the steps between them, and the `- 1` leaves out the two freeze days themselves.

For #5: the year's 65 freezing nights matched the normal, but they weren't spread out normally. January and December were colder than normal, and March and November were milder.

</details>

---
## A useful table

|**the loop way (Weeks 4–6)**|**the NumPy way (today)**|
|:-|:-|
|`count = 0`, then `if ...: count += 1`|`np.sum(a >= 90)`|
|`if ...:` + `.append()` to keep some values|`a[a >= 90]`|
|`and` / `or` / `not` inside the `if`|`&` / `\|` / `~`, with parentheses|
|`if x == -9999:` to fix bad values one at a time|`a[a == -9999] = np.nan`, on a `.copy()`|
|best-so-far loop to find **where** the max is|`a.argmax()` / `a.argmin()`|
|flag variable: "did this ever happen?"|`mask.any()`|
|flag variable: "did this always happen?"|`mask.all()`|
|streaks: "how many days **in a row**?"|still a loop! counter + best-so-far|